# Clean-Split ResNet-18 LAI Regression

This is a clean re-evaluation of ResNet-18 for continuous LAI regression. It uses only the verified leakage-free train, validation, and test manifests and does not modify or reuse the older ResNet-18 experiment.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from copy import deepcopy
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'lai_train_clean.csv').exists():
    for candidate in [project_root.parent, *project_root.parents]:
        if (candidate / 'data' / 'lai_train_clean.csv').exists():
            project_root = candidate
            break
DATA_DIR = project_root / 'data'
RESULTS_DIR = project_root / 'results'
MODELS_DIR = project_root / 'models'
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
print(f'Project root: {project_root}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Project root: C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water
Device: cpu


In [2]:
def resolve_image_path(stored_path: str) -> Path:
    path = Path(str(stored_path))
    if path.is_absolute():
        return path.resolve()
    # Resolve the path as stored in the manifest, relative to data/ first.
    candidates = [DATA_DIR / path, project_root / path, project_root.parent / path]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return (DATA_DIR / path).resolve()

def load_clean_split(filename: str) -> pd.DataFrame:
    frame = pd.read_csv(DATA_DIR / filename)
    required = {'image_path', 'LAI'}
    missing_columns = required - set(frame.columns)
    if missing_columns:
        raise ValueError(f'{filename} is missing columns: {sorted(missing_columns)}')
    frame = frame.copy()
    frame['image_path'] = frame['image_path'].astype(str)
    frame['LAI'] = pd.to_numeric(frame['LAI'], errors='coerce')
    frame['resolved_image_path'] = frame['image_path'].map(resolve_image_path)
    return frame

train_df = load_clean_split('lai_train_clean.csv')
val_df = load_clean_split('lai_val_clean.csv')
test_df = load_clean_split('lai_test_clean.csv')
all_frames = {'train': train_df, 'validation': val_df, 'test': test_df}
missing_paths = [(split, stored) for split, frame in all_frames.items() for stored, resolved in zip(frame['image_path'], frame['resolved_image_path']) if not resolved.is_file()]
invalid_targets = [(split, index) for split, frame in all_frames.items() for index, value in frame['LAI'].items() if not np.isfinite(value)]
print(f'Train samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Missing image paths: {len(missing_paths)}')
print(f'Invalid targets: {len(invalid_targets)}')
if missing_paths:
    raise FileNotFoundError(missing_paths[:5])
if invalid_targets:
    raise ValueError(invalid_targets[:5])

Train samples: 234
Validation samples: 51
Test samples: 51
Missing image paths: 0
Invalid targets: 0


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

identity_rows = []
for split_name, frame in all_frames.items():
    for _, row in frame.iterrows():
        identity_rows.append((split_name, str(row['resolved_image_path']), sha256_file(row['resolved_image_path'])))
identity_df = pd.DataFrame(identity_rows, columns=['split', 'resolved_path', 'sha256'])
cross_split_hashes = int((identity_df.groupby('sha256')['split'].nunique() > 1).sum())
cross_split_paths = int((identity_df.groupby('resolved_path')['split'].nunique() > 1).sum())
print(f'Cross-split identical image hashes: {cross_split_hashes}')
print(f'Cross-split duplicate paths: {cross_split_paths}')
if cross_split_hashes or cross_split_paths:
    raise ValueError('Clean split contains cross-split image overlap.')

Cross-split identical image hashes: 0
Cross-split duplicate paths: 0


In [4]:
class LAIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row['resolved_image_path']).convert('RGB')
        image = self.transform(image)
        target = torch.tensor(float(row['LAI']), dtype=torch.float32)
        return image, target, str(row['image_path'])

imagenet_normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    imagenet_normalize,
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    imagenet_normalize,
])
train_loader = DataLoader(LAIDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(LAIDataset(val_df, eval_transform), batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(LAIDataset(test_df, eval_transform), batch_size=32, shuffle=False, num_workers=0)
sample_images, _, _ = next(iter(train_loader))
print(f'Train batch shape: {tuple(sample_images.shape)}')

Train batch shape: (32, 3, 224, 224)


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0
    for images, targets, _ in loader:
        images, targets = images.to(device), targets.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            predictions = model(images).squeeze(1)
            loss = criterion(predictions, targets)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_items += images.size(0)
    return total_loss / total_items

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_state = None
best_epoch = 0
patience = 6
epochs_without_improvement = 0
max_epochs = 25
start_time = time.perf_counter()
for epoch in range(1, max_epochs + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}')
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
        print(f'Early stopping at epoch {epoch}.')
        break
training_seconds = time.perf_counter() - start_time
model.load_state_dict(best_state)
checkpoint_path = MODELS_DIR / 'lai_resnet18_clean_best.pth'
torch.save({'model_state_dict': model.state_dict(), 'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'seed': SEED}, checkpoint_path)
print(f'Best epoch: {best_epoch}; checkpoint: {checkpoint_path}')

Epoch 01 | train_loss=0.058513 | val_loss=0.055112
Epoch 02 | train_loss=0.059109 | val_loss=0.052519
Epoch 03 | train_loss=0.044381 | val_loss=0.042506
Epoch 04 | train_loss=0.021916 | val_loss=0.015623
Epoch 05 | train_loss=0.014449 | val_loss=0.031424
Epoch 06 | train_loss=0.014649 | val_loss=0.040455
Epoch 07 | train_loss=0.011078 | val_loss=0.024854
Epoch 08 | train_loss=0.008795 | val_loss=0.008695
Epoch 09 | train_loss=0.007339 | val_loss=0.008214
Epoch 10 | train_loss=0.006100 | val_loss=0.011566
Epoch 11 | train_loss=0.005795 | val_loss=0.004697
Epoch 12 | train_loss=0.005768 | val_loss=0.009120
Epoch 13 | train_loss=0.004025 | val_loss=0.007957
Epoch 14 | train_loss=0.003764 | val_loss=0.005936
Epoch 15 | train_loss=0.003431 | val_loss=0.006123
Epoch 16 | train_loss=0.003376 | val_loss=0.007797
Epoch 17 | train_loss=0.003172 | val_loss=0.005135
Early stopping at epoch 17.
Best epoch: 11; checkpoint: C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\models\lai_re

In [6]:
def predict(loader):
    model.eval()
    actual, predicted, paths = [], [], []
    with torch.no_grad():
        for images, targets, stored_paths in loader:
            predictions = model(images.to(device)).squeeze(1).cpu().numpy()
            actual.extend(targets.numpy().tolist())
            predicted.extend(predictions.tolist())
            paths.extend(stored_paths)
    return np.asarray(actual), np.asarray(predicted), paths

actual, predicted, stored_paths = predict(test_loader)
mae = float(mean_absolute_error(actual, predicted))
rmse = float(math.sqrt(mean_squared_error(actual, predicted)))
r2 = float(r2_score(actual, predicted))
predictions_df = pd.DataFrame({'image_path': stored_paths, 'actual_LAI': actual, 'predicted_LAI': predicted})
predictions_df.to_csv(RESULTS_DIR / 'lai_resnet18_clean_predictions.csv', index=False)

metrics = {
    'experiment': 'ResNet-18 LAI regression on verified clean split',
    'model': 'ResNet-18 ImageNet-pretrained with single-output regression layer',
    'MAE': mae, 'RMSE': rmse, 'R2': r2,
    'train_samples': int(len(train_df)), 'validation_samples': int(len(val_df)), 'test_samples': int(len(test_df)),
    'missing_image_paths': int(len(missing_paths)), 'invalid_targets': int(len(invalid_targets)),
    'cross_split_identical_image_hashes': cross_split_hashes, 'cross_split_duplicate_paths': cross_split_paths,
    'image_size': [224, 224], 'batch_size': 32, 'optimizer': 'Adam', 'learning_rate': 1e-4,
    'loss': 'SmoothL1Loss', 'seed': SEED, 'epochs_completed': len(train_losses),
    'best_epoch': int(best_epoch), 'best_validation_loss': float(best_val_loss), 'training_seconds': float(training_seconds),
}
(RESULTS_DIR / 'lai_resnet18_clean_metrics.json').write_text(json.dumps(metrics, indent=2))

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation loss')
plt.axvline(best_epoch, color='black', linestyle='--', alpha=0.6, label=f'Best epoch {best_epoch}')
plt.xlabel('Epoch'); plt.ylabel('SmoothL1 loss'); plt.title('Clean ResNet-18 Training and Validation Loss')
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lai_resnet18_clean_training_curve.png', dpi=200); plt.close()

plot_min, plot_max = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
padding = max((plot_max - plot_min) * 0.05, 0.001)
plt.figure(figsize=(6, 6))
plt.scatter(actual, predicted, alpha=0.75, s=38, color='#1769aa')
plt.plot([plot_min - padding, plot_max + padding], [plot_min - padding, plot_max + padding], 'k--', label='Ideal fit')
plt.xlabel('Actual LAI'); plt.ylabel('Predicted LAI'); plt.title('Actual vs Predicted LAI (Clean ResNet-18)')
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lai_resnet18_clean_actual_vs_predicted.png', dpi=200); plt.close()

plt.figure(figsize=(7, 4.5))
plt.hist(predicted - actual, bins=20, color='#d95f02', edgecolor='black')
plt.axvline(0, color='black', linestyle='--', linewidth=1.5)
plt.xlabel('Prediction error (predicted - actual)'); plt.ylabel('Count')
plt.title('Prediction Error Distribution (Clean ResNet-18)')
plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lai_resnet18_clean_error_distribution.png', dpi=200); plt.close()

print('FINAL VERIFICATION')
print(f'Train/validation/test: {len(train_df)} / {len(val_df)} / {len(test_df)}')
print(f'Missing image paths: {len(missing_paths)}')
print(f'Invalid targets: {len(invalid_targets)}')
print(f'MAE: {mae:.9f}')
print(f'RMSE: {rmse:.9f}')
print(f'R²: {r2:.9f}')
print('Clean ResNet-18 test metrics:', json.dumps({'MAE': mae, 'RMSE': rmse, 'R2': r2}))

FINAL VERIFICATION
Train/validation/test: 234 / 51 / 51
Missing image paths: 0
Invalid targets: 0
MAE: 0.096753566
RMSE: 0.120569790
R²: -19.543174905
Clean ResNet-18 test metrics: {"MAE": 0.09675356562194579, "RMSE": 0.12056978997235465, "R2": -19.543174904553112}


## Outputs

The notebook writes the clean ResNet-18 checkpoint, metrics JSON, test predictions, training curve, actual-vs-predicted plot, and error distribution under `models/` and `results/`. The test set is evaluated only after restoring the best validation-loss checkpoint.